## Cell 9 — Live Camera Test with Display (Google Colab)
- الكاميرا تظهر على الشاشة مباشرةً
- بيحدد نوع الغش (تليفون / التفات / طبيعي)
- بعد 35 ثانية بيبعت notification بالنتيجة

In [ ]:
# ============================================================
# إعدادات
# ============================================================
TEST_DURATION   = 35
CHEAT_THRESHOLD = 0.5
CAPTURE_EVERY   = 2        # صور frame كل كام ثانية
USE_MOBILE_CAM  = False
MOBILE_CAM_URL  = 'http://192.168.1.5:8080/video'

IMG_SIZE     = (224, 224)
CLASS_NAMES  = ['cheating', 'normal']

In [ ]:
# ============================================================
# Imports
# ============================================================
import numpy as np
import time
import base64
import cv2
import io
from PIL import Image
from IPython.display import display, Javascript, HTML, clear_output
from google.colab.output import eval_js
from tensorflow import keras
import ipywidgets as widgets

print('Imports done')

In [ ]:
# ============================================================
# JavaScript — يصور frame من المتصفح
# ============================================================
CAPTURE_JS = """
async function captureFrame() {
    const stream = await navigator.mediaDevices.getUserMedia({video: true});
    const video  = document.createElement('video');
    video.srcObject = stream;
    await video.play();
    await new Promise(r => setTimeout(r, 500));
    const canvas = document.createElement('canvas');
    canvas.width  = video.videoWidth;
    canvas.height = video.videoHeight;
    canvas.getContext('2d').drawImage(video, 0, 0);
    stream.getTracks().forEach(t => t.stop());
    return canvas.toDataURL('image/jpeg', 0.85);
}
captureFrame();
"""

def capture_from_browser():
    b64 = eval_js(CAPTURE_JS)
    if ',' in b64:
        b64 = b64.split(',')[1]
    img_bytes = base64.b64decode(b64)
    np_arr    = np.frombuffer(img_bytes, np.uint8)
    frame     = cv2.imdecode(np_arr, cv2.IMREAD_COLOR)
    return frame

def capture_from_mobile():
    cap = cv2.VideoCapture(MOBILE_CAM_URL)
    ret, frame = cap.read()
    cap.release()
    return frame if ret else None

print('Capture functions ready')

In [ ]:
# ============================================================
# Predict + تحديد نوع الغش
# ============================================================
def predict_frame(frame, loaded_model):
    """
    بيرجع:
      label      : 'cheating' أو 'normal'
      cheat_type : نوع الغش بناءً على تحليل الـ frame
      prob       : confidence
    """
    img = cv2.resize(frame, IMG_SIZE)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = img.astype(np.float32) / 255.0
    img = np.expand_dims(img, axis=0)

    prob  = loaded_model.predict(img, verbose=0)[0][0]
    label = CLASS_NAMES[int(prob > 0.5)]

    # ── تحديد نوع الغش بناءً على تحليل بسيط للـ frame ──
    cheat_type = None
    if label == 'cheating':
        cheat_type = detect_cheat_type(frame)

    return label, cheat_type, float(prob)


def detect_cheat_type(frame):
    """
    بيحاول يعرف نوع الغش:
    - Phone Use  : لو فيه مستطيل صغير رأسي (شكل التليفون)
    - Looking    : لو الوجه مش في المنتصف (التفات)
    """
    h, w = frame.shape[:2]

    # ── 1. تحقق لو فيه تليفون (مستطيل رأسي صغير) ──────
    gray    = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    edges   = cv2.Canny(blurred, 50, 150)
    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    for cnt in contours:
        x, y, cw, ch = cv2.boundingRect(cnt)
        area = cw * ch
        # التليفون: مستطيل رأسي مساحته معقولة
        if 2000 < area < (h * w * 0.15):
            ratio = ch / cw if cw > 0 else 0
            if 1.5 < ratio < 3.5:   # شكل التليفون
                return 'Phone Use'

    # ── 2. تحقق من التفات (وجه بعيد عن المنتصف) ────────
    face_cascade = cv2.CascadeClassifier(
        cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
    )
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5)

    if len(faces) > 0:
        fx, fy, fw, fh = faces[0]
        face_center_x = fx + fw // 2
        frame_center_x = w // 2
        offset = abs(face_center_x - frame_center_x) / w   # 0 = مركز, 1 = أقصى الجانب
        if offset > 0.25:   # الوجه بعيد عن المنتصف بأكتر من 25%
            return 'Looking Sideways'

    return 'Suspicious Behavior'

print('Prediction functions ready')

In [ ]:
# ============================================================
# رسم على الـ frame
# ============================================================
def draw_on_frame(frame, label, cheat_type, prob, remaining):
    out = frame.copy()
    h, w = out.shape[:2]

    is_cheat = (label == 'cheating')
    color     = (0, 0, 220)   if is_cheat else (0, 200, 0)
    bg_color  = (30, 0, 0)    if is_cheat else (0, 30, 0)

    # ── شريط علوي ────────────────────────────────────────
    cv2.rectangle(out, (0, 0), (w, 60), bg_color, -1)

    if is_cheat and cheat_type:
        status_text = f'CHEATING — {cheat_type}'
    elif is_cheat:
        status_text = 'CHEATING DETECTED'
    else:
        status_text = 'NORMAL'

    cv2.putText(out, status_text, (10, 38),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, color, 2)
    cv2.putText(out, f'{prob:.0%}', (w - 80, 38),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, color, 2)

    # ── شريط سفلي (الوقت المتبقي) ────────────────────────
    cv2.rectangle(out, (0, h - 35), (w, h), (20, 20, 20), -1)
    bar_w = int(w * remaining / TEST_DURATION)
    cv2.rectangle(out, (0, h - 35), (bar_w, h), (80, 80, 80), -1)
    cv2.putText(out, f'Time left: {remaining:.0f}s', (10, h - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.65, (200, 200, 200), 1)

    # ── لو غش: ارسم border أحمر ─────────────────────────
    if is_cheat:
        thickness = 6
        cv2.rectangle(out, (0, 0), (w, h), color, thickness)

        # ── ارسم label نوع الغش ─────────────────────────
        if cheat_type:
            tag_w, tag_h = 220, 36
            tag_x, tag_y = w // 2 - tag_w // 2, h // 2 - tag_h // 2
            cv2.rectangle(out, (tag_x, tag_y), (tag_x + tag_w, tag_y + tag_h),
                          (0, 0, 180), -1)
            cv2.rectangle(out, (tag_x, tag_y), (tag_x + tag_w, tag_y + tag_h),
                          (0, 0, 255), 2)
            cv2.putText(out, cheat_type, (tag_x + 10, tag_y + 24),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

    return out


def frame_to_jpeg_b64(frame):
    """يحول الـ frame لـ base64 عشان يعرضه في الـ notebook"""
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    pil_img   = Image.fromarray(frame_rgb)
    buffer    = io.BytesIO()
    pil_img.save(buffer, format='JPEG', quality=85)
    b64 = base64.b64encode(buffer.getvalue()).decode('utf-8')
    return b64

print('Drawing functions ready')

In [ ]:
# ============================================================
# Notification
# ============================================================
def send_notification(result, cheat_ratio, cheat_types_count):

    if result == 'CHEATING':
        color = '#cc0000'
        icon  = '⚠️'
        msg   = f'تم اكتشاف غش! ({cheat_ratio:.0%} من الوقت)'

        # تفاصيل نوع الغش
        details = ''
        for ctype, count in cheat_types_count.items():
            details += f'<div style="margin:4px 0">• {ctype}: {count} مرة</div>'
    else:
        color   = '#007700'
        icon    = '✅'
        msg     = f'لا يوجد غش ({1 - cheat_ratio:.0%} طبيعي)'
        details = ''

    display(HTML(f"""
    <div style="background:{color}; color:white; padding:20px;
                border-radius:12px; font-size:20px; font-weight:bold;
                text-align:center; margin:10px;">
        {icon}&nbsp; {msg}
        <div style="font-size:15px; margin-top:10px; font-weight:normal">
            {details}
        </div>
    </div>
    """))

    # صوت تنبيه
    if result == 'CHEATING':
        display(Javascript("""
            var ctx = new AudioContext();
            [0, 0.5, 1.0].forEach(function(t) {
                var o = ctx.createOscillator();
                o.connect(ctx.destination);
                o.frequency.value = 1000;
                o.start(ctx.currentTime + t);
                o.stop(ctx.currentTime + t + 0.3);
            });
        """))
    else:
        display(Javascript("""
            var ctx = new AudioContext();
            var o = ctx.createOscillator();
            o.connect(ctx.destination);
            o.frequency.value = 600;
            o.start(); o.stop(ctx.currentTime + 0.5);
        """))

print('Notification function ready')

In [ ]:
# ============================================================
# Main Camera Loop
# ============================================================
def run_camera_test():
    loaded_model = keras.models.load_model('saved_models/best_model.keras')
    print(f'Model loaded. بيصور لمدة {TEST_DURATION} ثانية...')
    print('اضغط على Stop Cell لو عايز توقف مبكراً')

    # Widget لعرض الـ frame
    img_widget = widgets.Image(format='jpeg', width=640)
    display(img_widget)

    start_time       = time.time()
    total_frames     = 0
    cheat_frames     = 0
    cheat_types_count = {}

    while True:
        elapsed   = time.time() - start_time
        remaining = TEST_DURATION - elapsed

        if elapsed >= TEST_DURATION:
            break

        # ── التقط frame ────────────────────────────────
        if USE_MOBILE_CAM:
            frame = capture_from_mobile()
        else:
            frame = capture_from_browser()

        if frame is None:
            print('فشل التقاط الـ frame — تأكد من الكاميرا')
            break

        # ── Predict ─────────────────────────────────────
        label, cheat_type, prob = predict_frame(frame, loaded_model)
        total_frames += 1

        if label == 'cheating':
            cheat_frames += 1
            if cheat_type:
                cheat_types_count[cheat_type] = cheat_types_count.get(cheat_type, 0) + 1

        # ── ارسم على الـ frame واعرضه ───────────────────
        annotated = draw_on_frame(frame, label, cheat_type, prob, remaining)
        img_widget.value = base64.b64decode(frame_to_jpeg_b64(annotated))

        time.sleep(CAPTURE_EVERY)

    # ── النتيجة النهائية ─────────────────────────────────
    if total_frames == 0:
        print('لم يتم التقاط أي frames!')
        return

    cheat_ratio  = cheat_frames / total_frames
    final_result = 'CHEATING' if cheat_ratio >= CHEAT_THRESHOLD else 'NORMAL'

    print(f'\nFrames analyzed : {total_frames}')
    print(f'Cheating frames : {cheat_frames} ({cheat_ratio:.0%})')
    if cheat_types_count:
        print(f'Cheat types     : {cheat_types_count}')

    send_notification(final_result, cheat_ratio, cheat_types_count)


run_camera_test()